In [ ]:
# Celula 1 - Setup Colab: clona e instala o pacote para evitar ModuleNotFoundError
import os, sys, subprocess, shutil
from pathlib import Path
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = "/content/TCC"
BRANCH = "update"
def _clone_repo(repo_url, repo_dir, branch=None):
    cmd = ["git", "clone"]
    if branch:
        cmd += ["--depth", "1", "--branch", branch]
    cmd += [repo_url, repo_dir]
    print('Executando:', ' '.join(cmd))
    return subprocess.run(cmd, check=True)
if IN_COLAB:
    try:
        if os.path.exists(REPO_DIR):
            print('Removendo pasta existente:', REPO_DIR)
            shutil.rmtree(REPO_DIR)
        try:
            _clone_repo(REPO_URL, REPO_DIR, BRANCH)
        except subprocess.CalledProcessError as exc:
            print(f'Clone com branch "{BRANCH}" falhou: {exc}. Tentando clone sem branch...')
            try:
                _clone_repo(REPO_URL, REPO_DIR, None)
            except subprocess.CalledProcessError as exc2:
                print('Clone git falhou. Tentando baixar ZIP do GitHub...')
                zip_url = REPO_URL.rstrip('.git') + f'/archive/refs/heads/{BRANCH}.zip'
                try:
                    import urllib.request, zipfile, io
                    print('Baixando', zip_url)
                    data = urllib.request.urlopen(zip_url).read()
                    z = zipfile.ZipFile(io.BytesIO(data))
                    z.extractall('/content')
                    # a extração cria uma pasta com sufixo de branch - detecta e renomeia
                    extracted = next(p for p in Path('/content').iterdir() if p.is_dir() and p.name.startswith(Path(REPO_URL).stem))
                    extracted.rename(REPO_DIR)
                except Exception as exc3:
                    print('Falha ao baixar/extrair ZIP:', exc3)
                    raise RuntimeError('Não foi possível obter o repositório.') from exc3
        os.chdir(REPO_DIR)
        print('Instalando pacote em modo editável...')
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
        req = os.path.join(REPO_DIR, 'requirements.txt')
        if os.path.exists(req):
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", req], check=True)
        src = os.path.join(REPO_DIR, 'src')
        if os.path.isdir(src) and src not in sys.path:
            sys.path.insert(0, src)
        print('Setup concluído. Reinicie o runtime se necessário.')
    except Exception as e:
        print('Erro no setup do repositório:', e)
        raise
else:
    print('Não detectado Colab. Verifique se está executando localmente.')